# Survey Data Quality Checker
> **How to use:**  
> 1. Upload your `.xlsx` or `.csv` file when the upload cell runs  
> 2. Review the **AUTO-DETECTED COLUMNS** section and adjust if needed  
> 3. Adjust **THRESHOLDS** if needed  
> 4. Run all cells — a styled Excel report will download automatically

In [ ]:
# ── CELL 1: Install dependencies ─────────────────────────────────────────────
import subprocess, sys
for pkg in ["openpyxl", "pandas", "numpy"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

In [ ]:
# ── CELL 2: Imports ───────────────────────────────────────────────────────────
import re, math, warnings
from collections import defaultdict
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import openpyxl
from openpyxl.styles import (
    PatternFill, Font, Alignment, Border, Side, GradientFill
)
from openpyxl.utils import get_column_letter

warnings.filterwarnings("ignore")

In [ ]:
# ── CELL 3: Upload / Load file ────────────────────────────────────────────────
try:
    from google.colab import files as colab_files
    print("📂  Upload your survey data file (.xlsx or .csv)…")
    uploaded = colab_files.upload()
    FILE_PATH = next(iter(uploaded))            # first uploaded file
    print(f"✅  Loaded: {FILE_PATH}")
except Exception:
    # Running locally — set FILE_PATH manually
    FILE_PATH = "your_survey_data.xlsx"        # ← change this if running locally
    print(f"ℹ  Using local file: {FILE_PATH}")

## ⚙️ Configuration
Adjust thresholds and column mappings below before running checks.

In [ ]:
# ── CELL 4: CONFIGURATION ─────────────────────────────────────────────────────
# ─────────────────────────────────────────────────────────────────────────────
#  THRESHOLDS  (change these to match your field protocol)
# ─────────────────────────────────────────────────────────────────────────────
THRESH = dict(
    short_min   = 20,    # interviews shorter than this (minutes) are flagged
    long_hr     = 3,     # interviews longer than this (hours) are flagged
    extreme_hr  = 6,     # interviews longer than this (hours) are extreme
    sl_flagged  = 6,     # ≥ this many SL batteries → Flagged
    sl_suspected= 4,     # ≥ this many SL batteries → Suspected
    gps_radius_km = 200, # GPS point must be within this distance of state centroid
)

# ─────────────────────────────────────────────────────────────────────────────
#  AUTO-DETECTED COLUMNS  (override by setting string values below)
#  Leave as None to auto-detect from the column headers
# ─────────────────────────────────────────────────────────────────────────────
COL = dict(
    start       = None,   # interview start datetime column
    end         = None,   # interview end datetime column
    enumerator  = None,   # enumerator / interviewer name
    respondent  = None,   # respondent name
    phone       = None,   # respondent phone number (CATI/CAPI recall)
    key         = None,   # unique record ID / instanceID
    state       = None,   # state or location name
    lga         = None,   # LGA (optional)
    gps_lat     = None,   # GPS latitude
    gps_lon     = None,   # GPS longitude
)

# ─────────────────────────────────────────────────────────────────────────────
#  OUTPUT
# ─────────────────────────────────────────────────────────────────────────────
OUTPUT_FILE = "Survey_QC_Report.xlsx"

In [ ]:
# ── CELL 5: Load data ─────────────────────────────────────────────────────────
ext = Path(FILE_PATH).suffix.lower()
if ext == ".csv":
    df = pd.read_csv(FILE_PATH, dtype=str, keep_default_na=False)
else:
    df = pd.read_excel(FILE_PATH, dtype=str, keep_default_na=False)

df.columns = [str(c).strip() for c in df.columns]
print(f"📊  {len(df):,} rows × {len(df.columns)} columns loaded")

In [ ]:
# ── CELL 6: Auto-detect columns ───────────────────────────────────────────────
HINTS = {
    "start":       re.compile(r"start|begin", re.I),
    "end":         re.compile(r"\bend\b|finish|endtime", re.I),
    "enumerator":  re.compile(r"enum|interviewer|field.?worker|your.?name|collector", re.I),
    "respondent":  re.compile(r"respondent|participant|subject", re.I),
    "phone":       re.compile(r"phone|mobile|tel|contact|number", re.I),
    "key":         re.compile(r"\bkey\b|\bid\b|uuid|instanceid", re.I),
    "state":       re.compile(r"\bstate\b|region|province", re.I),
    "lga":         re.compile(r"\blga\b|local.?gov", re.I),
    "gps_lat":     re.compile(r"lat", re.I),
    "gps_lon":     re.compile(r"\blon\b|\blng\b|longitude", re.I),
}

for role, hint in HINTS.items():
    if COL[role] is None:
        match = next((c for c in df.columns if hint.search(c)), None)
        COL[role] = match

print("\n🔍  Column mapping:")
for role, col in COL.items():
    print(f"    {role:<14} → {col or '(not found)'}")

## 🔧 Helper Functions
Datetime parsing, duration formatting, and Haversine distance.

In [ ]:
# ── CELL 7: Datetime parser ───────────────────────────────────────────────────
MONTHS = {"jan":1,"feb":2,"mar":3,"apr":4,"may":5,"jun":6,
          "jul":7,"aug":8,"sep":9,"oct":10,"nov":11,"dec":12}

FORMATS = [
    "%a %b %d %Y %H:%M:%S",
    "%a %b %d %Y %H:%M",
    "%b %d, %Y %H:%M:%S",
    "%b %d, %Y %H:%M",
    "%b %d %Y %H:%M:%S",
    "%Y-%m-%d %H:%M:%S",
    "%Y-%m-%dT%H:%M:%S",
    "%d/%m/%Y %H:%M:%S",
    "%d/%m/%Y %H:%M",
]

def parse_dt(val):
    if pd.isna(val) or str(val).strip() == "":
        return None
    s = str(val).strip()
    s = re.sub(r"\s+G(MT)?(\+\d+)?$", "", s).strip()
    for fmt in FORMATS:
        try:
            return datetime.strptime(s, fmt)
        except ValueError:
            pass
    try:
        return pd.to_datetime(s, dayfirst=True, infer_datetime_format=True).to_pydatetime()
    except Exception:
        return None

def fmt_dur(td):
    if td is None:
        return "—"
    total_s = int(abs(td.total_seconds()))
    h, rem  = divmod(total_s, 3600)
    m, s    = divmod(rem, 60)
    sign    = "-" if td.total_seconds() < 0 else ""
    return f"{sign}{h}h {m:02d}m {s:02d}s"

## 🕐 Duration Analysis

In [ ]:
# ── CELL 8: DURATION ANALYSIS ─────────────────────────────────────────────────
def analyze_duration(df):
    short_td   = timedelta(minutes=THRESH["short_min"])
    long_td    = timedelta(hours=THRESH["long_hr"])
    extreme_td = timedelta(hours=THRESH["extreme_hr"])

    records = {"negative": [], "short": [], "long": [], "extreme": [], "noparse": []}

    for idx, row in df.iterrows():
        row_num = idx + 2
        s_raw = row.get(COL["start"], "") if COL["start"] else ""
        e_raw = row.get(COL["end"],   "") if COL["end"]   else ""

        s_dt = parse_dt(s_raw)
        e_dt = parse_dt(e_raw)

        base = {
            "Row":         row_num,
            "Key":         row.get(COL["key"],        "") if COL["key"]        else "",
            "Enumerator":  row.get(COL["enumerator"], "") if COL["enumerator"] else "",
            "Respondent":  row.get(COL["respondent"], "") if COL["respondent"] else "",
            "Phone":       row.get(COL["phone"],      "") if COL["phone"]      else "",
            "Start Raw":   str(s_raw),
            "End Raw":     str(e_raw),
        }

        if not s_dt or not e_dt:
            records["noparse"].append({**base, "Issue": "Could not parse timestamp"})
            continue

        td = e_dt - s_dt
        base["Start"]    = s_dt.strftime("%Y-%m-%d %H:%M:%S")
        base["End"]      = e_dt.strftime("%Y-%m-%d %H:%M:%S")
        base["Duration"] = fmt_dur(td)
        base["Duration_min"] = round(td.total_seconds() / 60, 1)

        if td.total_seconds() < 0:
            records["negative"].append({**base, "Flag": "Negative Duration"})
        elif td < short_td:
            records["short"].append({**base, "Flag": f"Short (<{THRESH['short_min']} min)"})
        elif td >= extreme_td:
            records["extreme"].append({**base, "Flag": f"Extreme (>{THRESH['extreme_hr']} hrs)"})
        elif td >= long_td:
            records["long"].append({**base, "Flag": f"Long (>{THRESH['long_hr']} hrs)"})

    return records

## 📋 Straightlining Analysis

In [ ]:
# ── CELL 9: STRAIGHTLINING ANALYSIS ──────────────────────────────────────────
LIKERT = {1, 2, 3, 4, 5}

def analyze_straightlining(df):
    q_groups = defaultdict(list)
    for col in df.columns:
        m = re.match(r"^(Q\d+)_", col, re.I)
        if m:
            q_groups[m.group(1).upper()].append(col)
    batteries = [q for q, cols in q_groups.items() if len(cols) >= 2]
    batteries.sort(key=lambda x: int(re.search(r"\d+", x).group()))
    n_batt = len(batteries)

    flagged, suspected, clean = [], [], []

    for idx, row in df.iterrows():
        row_num = idx + 2
        sl_batteries = []
        sl_values    = {}

        for q in batteries:
            vals = []
            for col in q_groups[q]:
                try:
                    v = int(float(str(row[col]).strip()))
                    if v in LIKERT:
                        vals.append(v)
                except (ValueError, TypeError):
                    pass
            if len(vals) >= 2 and len(set(vals)) == 1:
                sl_batteries.append(q)
                sl_values[q] = vals[0]

        n_sl = len(sl_batteries)
        rate = round(100 * n_sl / n_batt, 1) if n_batt else 0.0

        if n_sl >= THRESH["sl_flagged"]:
            verdict = "Flagged — Straightlining"
        elif n_sl >= THRESH["sl_suspected"]:
            verdict = "Suspected — Needs Review"
        else:
            verdict = "Clean — No Flag"

        rec = {
            "Row":         row_num,
            "Key":         row.get(COL["key"],        "") if COL["key"]        else "",
            "Enumerator":  row.get(COL["enumerator"], "") if COL["enumerator"] else "",
            "Respondent":  row.get(COL["respondent"], "") if COL["respondent"] else "",
            "Phone":       row.get(COL["phone"],      "") if COL["phone"]      else "",
            "SL Batteries":n_sl,
            "Total Batt":  n_batt,
            "SL Rate (%)": rate,
            "Batteries SL'd": ", ".join(sl_batteries),
            "Verdict":     verdict,
        }
        if verdict == "Flagged — Straightlining":
            flagged.append(rec)
        elif verdict == "Suspected — Needs Review":
            suspected.append(rec)
        else:
            clean.append(rec)

    enum_summary = defaultdict(lambda: {"Enumerator": "", "Total": 0, "Flagged": 0, "Suspected": 0, "Clean": 0})
    for rec in flagged + suspected + clean:
        en = rec["Enumerator"] or "(unknown)"
        enum_summary[en]["Enumerator"] = en
        enum_summary[en]["Total"] += 1
        if rec["Verdict"] == "Flagged — Straightlining":
            enum_summary[en]["Flagged"] += 1
        elif rec["Verdict"] == "Suspected — Needs Review":
            enum_summary[en]["Suspected"] += 1
        else:
            enum_summary[en]["Clean"] += 1

    for en in enum_summary:
        t = enum_summary[en]["Total"]
        enum_summary[en]["Flag Rate (%)"] = round(
            100 * (enum_summary[en]["Flagged"] + enum_summary[en]["Suspected"]) / t, 1) if t else 0

    enum_list = sorted(enum_summary.values(),
                       key=lambda x: -(x["Flagged"] + x["Suspected"]))

    return dict(flagged=flagged, suspected=suspected, clean=clean,
                batteries=batteries, n_batt=n_batt, enum_summary=enum_list)

## 📍 GPS Coherence Analysis

In [ ]:
# ── CELL 10: GPS ANALYSIS ─────────────────────────────────────────────────────
NG_BOUNDS = dict(lat_min=3.5, lat_max=13.9, lon_min=2.7, lon_max=14.8)

NG_STATES = {
    "abia":{"lat":5.45,"lon":7.52},"adamawa":{"lat":9.33,"lon":12.40},
    "akwa ibom":{"lat":5.01,"lon":7.93},"anambra":{"lat":6.22,"lon":6.94},
    "bauchi":{"lat":10.31,"lon":9.84},"bayelsa":{"lat":4.77,"lon":6.07},
    "benue":{"lat":7.34,"lon":8.74},"borno":{"lat":11.85,"lon":13.16},
    "cross river":{"lat":5.87,"lon":8.60},"delta":{"lat":5.50,"lon":5.90},
    "ebonyi":{"lat":6.24,"lon":8.01},"edo":{"lat":6.34,"lon":5.63},
    "ekiti":{"lat":7.72,"lon":5.31},"enugu":{"lat":6.46,"lon":7.55},
    "gombe":{"lat":10.29,"lon":11.17},"imo":{"lat":5.57,"lon":7.06},
    "jigawa":{"lat":12.22,"lon":9.56},"kaduna":{"lat":10.52,"lon":7.44},
    "kano":{"lat":12.00,"lon":8.52},"katsina":{"lat":12.99,"lon":7.62},
    "kebbi":{"lat":11.49,"lon":4.21},"kogi":{"lat":7.80,"lon":6.74},
    "kwara":{"lat":8.97,"lon":4.39},"lagos":{"lat":6.52,"lon":3.38},
    "nasarawa":{"lat":8.50,"lon":8.22},"niger":{"lat":9.93,"lon":5.60},
    "ogun":{"lat":6.99,"lon":3.47},"ondo":{"lat":6.91,"lon":5.15},
    "osun":{"lat":7.56,"lon":4.56},"oyo":{"lat":7.85,"lon":3.93},
    "plateau":{"lat":9.22,"lon":9.52},"rivers":{"lat":4.82,"lon":6.92},
    "sokoto":{"lat":13.06,"lon":5.24},"taraba":{"lat":7.99,"lon":10.77},
    "yobe":{"lat":12.29,"lon":11.44},"zamfara":{"lat":12.16,"lon":6.22},
    "fct":{"lat":9.07,"lon":7.40},"abuja":{"lat":9.07,"lon":7.40},
    "federal capital territory":{"lat":9.07,"lon":7.40},
}

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1))*math.cos(math.radians(lat2))*math.sin(dlon/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

def state_centroid(val):
    s = str(val).lower().strip()
    for k, v in NG_STATES.items():
        if k in s or s in k:
            return k, v["lat"], v["lon"]
    return None, None, None

def analyze_gps(df):
    if not COL["gps_lat"] or not COL["gps_lon"]:
        print("ℹ  GPS columns not mapped — skipping GPS check")
        return {"issues": [], "skipped": True}

    issues = []
    enum_gps = defaultdict(list)

    for idx, row in df.iterrows():
        row_num = idx + 2
        lat_raw = row.get(COL["gps_lat"], "")
        lon_raw = row.get(COL["gps_lon"], "")
        state_raw = row.get(COL["state"], "") if COL["state"] else ""
        phone_val  = row.get(COL["phone"], "") if COL["phone"] else ""

        base = {
            "Row":        row_num,
            "Key":        row.get(COL["key"],        "") if COL["key"]        else "",
            "Enumerator": row.get(COL["enumerator"], "") if COL["enumerator"] else "",
            "Respondent": row.get(COL["respondent"], "") if COL["respondent"] else "",
            "Phone":      phone_val,
            "State":      state_raw,
            "Lat Raw":    str(lat_raw),
            "Lon Raw":    str(lon_raw),
        }

        flags = []
        if str(lat_raw).strip() == "" or str(lon_raw).strip() == "":
            flags.append("Missing GPS")
            issues.append({**base, "Issue": " | ".join(flags), "Severity": "High"})
            continue

        try:
            lat = float(lat_raw)
            lon = float(lon_raw)
        except ValueError:
            flags.append("Invalid GPS format")
            issues.append({**base, "Issue": " | ".join(flags), "Severity": "High"})
            continue

        base["Latitude"]  = lat
        base["Longitude"] = lon

        if lat == 0.0 and lon == 0.0:
            flags.append("Null Island (0,0)")
        if not (NG_BOUNDS["lat_min"] <= lat <= NG_BOUNDS["lat_max"] and
                NG_BOUNDS["lon_min"] <= lon <= NG_BOUNDS["lon_max"]):
            flags.append(f"Outside Nigeria ({lat:.4f}, {lon:.4f})")

        if state_raw:
            sk, slat, slon = state_centroid(state_raw)
            if sk:
                dist = haversine_km(lat, lon, slat, slon)
                if dist > THRESH["gps_radius_km"]:
                    flags.append(f"{dist:.0f} km from '{sk}' centroid (>{THRESH['gps_radius_km']} km)")

        if flags:
            severity = "High" if any("Outside" in f or "Null" in f for f in flags) else "Medium"
            issues.append({**base, "Issue": " | ".join(flags), "Severity": severity})

        en = base["Enumerator"] or "(unknown)"
        enum_gps[en].append((row_num, lat, lon))

    for en, pts in enum_gps.items():
        freq = defaultdict(list)
        for rn, lat, lon in pts:
            freq[f"{lat:.4f},{lon:.4f}"].append(rn)
        for coords, rows_list in freq.items():
            if len(rows_list) >= 4:
                clat, clon = map(float, coords.split(","))
                issues.append({
                    "Row":        f"Rows: {', '.join(str(r) for r in rows_list[:8])}{'…' if len(rows_list)>8 else ''}",
                    "Key":        "",
                    "Enumerator": en,
                    "Respondent": "",
                    "Phone":      "",
                    "State":      "",
                    "Lat Raw":    str(clat),
                    "Lon Raw":    str(clon),
                    "Latitude":   clat,
                    "Longitude":  clon,
                    "Issue":      f"GPS Fixation: same location used {len(rows_list)} times",
                    "Severity":   "High",
                })

    return {"issues": issues, "skipped": False}

## 🔁 Duplicate Detection

In [ ]:
# ── CELL 11: DUPLICATE ANALYSIS ───────────────────────────────────────────────
def analyze_duplicates(df):
    dups = []
    seen = set()

    if COL["key"]:
        key_groups = defaultdict(list)
        for idx, row in df.iterrows():
            k = str(row.get(COL["key"], "")).strip()
            if k:
                key_groups[k].append(idx + 2)
        for k, rows_list in key_groups.items():
            if len(rows_list) > 1:
                for rn in rows_list:
                    if rn not in seen:
                        seen.add(rn)
                        row = df.iloc[rn - 2]
                        dups.append({
                            "Row":          rn,
                            "Key":          k,
                            "Enumerator":   row.get(COL["enumerator"], "") if COL["enumerator"] else "",
                            "Respondent":   row.get(COL["respondent"], "") if COL["respondent"] else "",
                            "Phone":        row.get(COL["phone"], "")      if COL["phone"]      else "",
                            "Duplicate Type": "Duplicate KEY",
                            "Group Size":   len(rows_list),
                        })

    if COL["respondent"] or COL["enumerator"]:
        sig_groups = defaultdict(list)
        for idx, row in df.iterrows():
            resp  = str(row.get(COL["respondent"], "")).strip() if COL["respondent"] else ""
            enum  = str(row.get(COL["enumerator"], "")).strip() if COL["enumerator"] else ""
            phone = str(row.get(COL["phone"], "")).strip()      if COL["phone"]      else ""
            sig   = f"{resp}|{enum}|{phone}"
            if resp or enum:
                sig_groups[sig].append(idx + 2)
        for sig, rows_list in sig_groups.items():
            if len(rows_list) > 1:
                for rn in rows_list:
                    if rn not in seen:
                        seen.add(rn)
                        row = df.iloc[rn - 2]
                        dups.append({
                            "Row":          rn,
                            "Key":          row.get(COL["key"], "") if COL["key"] else "",
                            "Enumerator":   row.get(COL["enumerator"], "") if COL["enumerator"] else "",
                            "Respondent":   row.get(COL["respondent"], "") if COL["respondent"] else "",
                            "Phone":        row.get(COL["phone"], "")      if COL["phone"]      else "",
                            "Duplicate Type": "Same Respondent+Enumerator",
                            "Group Size":   len(rows_list),
                        })

    dups.sort(key=lambda x: x["Row"] if isinstance(x["Row"], int) else 0)
    return dups

## ▶️ Run All Checks

In [ ]:
# ── CELL 12: RUN ALL CHECKS ───────────────────────────────────────────────────
print("\n⏳  Running quality checks…")
dur_results = analyze_duration(df)
sl_results  = analyze_straightlining(df)
gps_results = analyze_gps(df)
dup_results = analyze_duplicates(df)

total_dur = sum(len(dur_results[k]) for k in ["negative","short","long","extreme"])
print(f"   ✔ Duration issues:    {total_dur}")
print(f"   ✔ Straightlining:     {len(sl_results['flagged'])} flagged, {len(sl_results['suspected'])} suspected")
print(f"   ✔ GPS issues:         {len(gps_results['issues'])}")
print(f"   ✔ Duplicates:         {len(dup_results)}")
print(f"   ✔ Batteries detected: {sl_results['n_batt']} ({', '.join(sl_results['batteries'])})")

## 📊 Build Excel Report

In [ ]:
# ── CELL 13: BUILD EXCEL REPORT ───────────────────────────────────────────────
RED_FILL    = PatternFill("solid", fgColor="FFB3B3")
AMBER_FILL  = PatternFill("solid", fgColor="FFE5B4")
GREEN_FILL  = PatternFill("solid", fgColor="D6F5E3")
PURPLE_FILL = PatternFill("solid", fgColor="E8D5FF")
BLUE_FILL   = PatternFill("solid", fgColor="D0E8FF")
GREY_FILL   = PatternFill("solid", fgColor="EBEBEB")
HEADER_FILL = PatternFill("solid", fgColor="1D4ED8")

BOLD    = Font(bold=True)
WHITE_B = Font(bold=True, color="FFFFFF")
RED_F   = Font(color="7F0000", bold=True)
AMBER_F = Font(color="7A4000", bold=True)
GREEN_F = Font(color="065F46", bold=True)
GREY_F  = Font(color="374151")

CENTER  = Alignment(horizontal="center", vertical="center")
LEFT    = Alignment(horizontal="left",   vertical="center")
WRAP    = Alignment(wrap_text=True, vertical="top")

def thin_border():
    s = Side(style="thin", color="CCCCCC")
    return Border(left=s, right=s, top=s, bottom=s)

def write_df_to_sheet(ws, rows, col_fills=None, row_fill_col=None, row_fill_map=None):
    if not rows:
        ws.cell(1, 1, "No issues found ✓").font = Font(color="065F46", bold=True)
        return
    headers = list(rows[0].keys())
    for ci, h in enumerate(headers, 1):
        cell = ws.cell(1, ci, h)
        cell.fill  = HEADER_FILL
        cell.font  = WHITE_B
        cell.alignment = CENTER
        cell.border = thin_border()
    ws.row_dimensions[1].height = 20
    for ri, rec in enumerate(rows, 2):
        rf = None
        if row_fill_col and row_fill_map:
            key = rec.get(row_fill_col, "")
            rf = row_fill_map.get(key)
        for ci, h in enumerate(headers, 1):
            val = rec.get(h, "")
            cell = ws.cell(ri, ci, val)
            cell.border    = thin_border()
            cell.alignment = LEFT
            if rf:
                cell.fill = rf
            elif col_fills and h in col_fills:
                cell.fill = col_fills[h]
    for ci, h in enumerate(headers, 1):
        max_w = max(len(str(h)), *(len(str(rec.get(h, ""))) for rec in rows))
        ws.column_dimensions[get_column_letter(ci)].width = min(max_w + 3, 40)
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

def style_sheet_header(ws, title, subtitle, colour="1D4ED8"):
    ws.insert_rows(1, 3)
    ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=10)
    t = ws.cell(1, 1, title)
    t.font      = Font(bold=True, size=13, color="FFFFFF")
    t.fill      = PatternFill("solid", fgColor=colour)
    t.alignment = CENTER
    ws.row_dimensions[1].height = 24
    ws.merge_cells(start_row=2, start_column=1, end_row=2, end_column=10)
    s = ws.cell(2, 1, subtitle)
    s.font      = Font(size=10, color="4B5563")
    s.fill      = PatternFill("solid", fgColor="EFF4FF")
    s.alignment = LEFT
    ws.row_dimensions[2].height = 16
    ws.row_dimensions[3].height = 4

wb = openpyxl.Workbook()
ws_sum = wb.active
ws_sum.title = "QC Summary"

ws_sum.merge_cells("A1:H1")
tc = ws_sum["A1"]
tc.value = f"Survey Data Quality Report  —  NOI Polls  —  Generated {datetime.now():%d %b %Y %H:%M}"
tc.font  = Font(bold=True, size=13, color="FFFFFF")
tc.fill  = HEADER_FILL
tc.alignment = CENTER
ws_sum.row_dimensions[1].height = 26

ws_sum.merge_cells("A2:H2")
ws_sum["A2"] = f"Source file: {FILE_PATH}   |   {len(df):,} records   |   {sl_results['n_batt']} Likert batteries: {', '.join(sl_results['batteries'])}"
ws_sum["A2"].font = Font(size=10, color="4B5563")
ws_sum["A2"].fill = PatternFill("solid", fgColor="EFF4FF")
ws_sum["A2"].alignment = LEFT
ws_sum.row_dimensions[2].height = 16

kpis = [
    ("Total Records",    len(df),                              "1D4ED8", "FFFFFF"),
    ("Duration Issues",  total_dur,                            "B45309", "FFFFFF"),
    ("SL Flagged",       len(sl_results["flagged"]),           "DC2626", "FFFFFF"),
    ("SL Suspected",     len(sl_results["suspected"]),         "D97706", "FFFFFF"),
    ("GPS Issues",       len(gps_results["issues"]),           "6D28D9", "FFFFFF"),
    ("Duplicates",       len(dup_results),                     "374151", "FFFFFF"),
    ("Negative Dur",     len(dur_results["negative"]),         "DC2626", "FFFFFF"),
    ("Extreme Dur",      len(dur_results["extreme"]),          "B45309", "FFFFFF"),
]
ws_sum.row_dimensions[4].height = 14
ws_sum.row_dimensions[5].height = 30
ws_sum.row_dimensions[6].height = 20

for ci, (label, val, bg, fg) in enumerate(kpis, 1):
    ws_sum.cell(4, ci, label).font  = Font(bold=True, size=9, color="374151")
    ws_sum.cell(4, ci).alignment = CENTER
    v = ws_sum.cell(5, ci, val)
    v.font      = Font(bold=True, size=18, color=fg)
    v.fill      = PatternFill("solid", fgColor=bg)
    v.alignment = CENTER
    v.border    = thin_border()
    ws_sum.column_dimensions[get_column_letter(ci)].width = 17

start_row = 8
ws_sum.cell(start_row, 1, "Enumerator Straightlining Summary").font = Font(bold=True, size=11)
start_row += 1

enum_headers = ["Enumerator", "Total Interviews", "Flagged", "Suspected", "Clean", "Flag Rate (%)"]
for ci, h in enumerate(enum_headers, 1):
    c = ws_sum.cell(start_row, ci, h)
    c.fill = HEADER_FILL; c.font = WHITE_B; c.alignment = CENTER; c.border = thin_border()

for ri, rec in enumerate(sl_results["enum_summary"], start_row + 1):
    rf = RED_FILL if rec["Flagged"] > 0 else (AMBER_FILL if rec["Suspected"] > 0 else GREEN_FILL)
    ws_sum.cell(ri, 1, rec["Enumerator"]).border   = thin_border()
    ws_sum.cell(ri, 2, rec["Total"]).border         = thin_border()
    ws_sum.cell(ri, 3, rec["Flagged"]).border       = thin_border()
    ws_sum.cell(ri, 4, rec["Suspected"]).border     = thin_border()
    ws_sum.cell(ri, 5, rec["Clean"]).border         = thin_border()
    ws_sum.cell(ri, 6, rec["Flag Rate (%)"]).border = thin_border()
    for ci in range(1, 7):
        ws_sum.cell(ri, ci).fill      = rf
        ws_sum.cell(ri, ci).alignment = CENTER

ws_sum.freeze_panes = "A3"

FILL_MAP_SL  = {
    "Flagged — Straightlining":  RED_FILL,
    "Suspected — Needs Review":  AMBER_FILL,
    "Clean — No Flag":           GREEN_FILL,
}
FILL_MAP_GPS = {"High": RED_FILL, "Medium": AMBER_FILL}
FILL_MAP_DUR = {
    "Negative Duration":                          RED_FILL,
    f"Short (<{THRESH['short_min']} min)":        AMBER_FILL,
    f"Long (>{THRESH['long_hr']} hrs)":           AMBER_FILL,
    f"Extreme (>{THRESH['extreme_hr']} hrs)":     RED_FILL,
}

def add_sheet(wb, name, rows, title, subtitle, fill_col=None, fill_map=None, title_colour="1D4ED8"):
    ws = wb.create_sheet(name)
    write_df_to_sheet(ws, rows, row_fill_col=fill_col, row_fill_map=fill_map)
    style_sheet_header(ws, title, subtitle, colour=title_colour)
    return ws

add_sheet(wb, "Negative Duration",  dur_results["negative"],
    "Negative Duration Interviews",
    "End time is before start time — likely device clock error or date entry mistake",
    "Flag", FILL_MAP_DUR, "DC2626")

add_sheet(wb, "Short Duration", dur_results["short"],
    f"Short Duration (<{THRESH['short_min']} min)",
    f"Interviews completed in under {THRESH['short_min']} minutes — may indicate skipped sections",
    "Flag", FILL_MAP_DUR, "B45309")

all_long = dur_results["long"] + dur_results["extreme"]
all_long.sort(key=lambda x: -x["Duration_min"])
add_sheet(wb, "Long Duration", all_long,
    f"Long Duration (>{THRESH['long_hr']} hrs)",
    f"Interviews exceeding {THRESH['long_hr']} hours — review for data entry delays",
    "Flag", FILL_MAP_DUR, "B45309")

add_sheet(wb, "SL Flagged", sl_results["flagged"],
    "Straightlining — Flagged",
    f"Respondents giving identical answers across ≥{THRESH['sl_flagged']} of {sl_results['n_batt']} batteries",
    "Verdict", FILL_MAP_SL, "DC2626")

add_sheet(wb, "SL Suspected", sl_results["suspected"],
    "Straightlining — Suspected",
    f"Respondents with identical answers across {THRESH['sl_suspected']}–{THRESH['sl_flagged']-1} batteries — needs review",
    "Verdict", FILL_MAP_SL, "D97706")

add_sheet(wb, "GPS Issues", gps_results["issues"],
    "GPS Coherence Issues",
    "Records with missing, invalid, or geographically inconsistent GPS coordinates",
    "Severity", FILL_MAP_GPS, "6D28D9")

add_sheet(wb, "Duplicates", dup_results,
    "Potential Duplicate Records",
    "Records sharing the same unique key or the same respondent + enumerator combination",
    title_colour="374151")

if dur_results["noparse"]:
    add_sheet(wb, "Timestamp Errors", dur_results["noparse"],
        "Unparseable Timestamps",
        "Rows where start or end time could not be parsed — check column mapping or data format",
        title_colour="991B1B")

print("✅  Excel workbook built.")

## 💾 Save & Download Report

In [ ]:
# ── CELL 14: SAVE & DOWNLOAD ──────────────────────────────────────────────────
wb.save(OUTPUT_FILE)
print(f"\n✅  Report saved: {OUTPUT_FILE}")
print(f"    Sheets: {[ws.title for ws in wb.worksheets]}")

try:
    from google.colab import files as colab_files
    colab_files.download(OUTPUT_FILE)
    print("📥  Downloading to your computer…")
except Exception:
    print(f"ℹ  File saved locally as: {OUTPUT_FILE}")

print("\n── QC SUMMARY ──────────────────────────────────────────")
print(f"  Total records       : {len(df):,}")
print(f"  Batteries detected  : {sl_results['n_batt']}  ({', '.join(sl_results['batteries'])})")
print(f"  Negative duration   : {len(dur_results['negative'])}")
print(f"  Short duration      : {len(dur_results['short'])}")
print(f"  Long duration       : {len(dur_results['long'])}")
print(f"  Extreme duration    : {len(dur_results['extreme'])}")
print(f"  SL Flagged          : {len(sl_results['flagged'])}")
print(f"  SL Suspected        : {len(sl_results['suspected'])}")
print(f"  GPS issues          : {len(gps_results['issues'])}")
print(f"  Duplicates          : {len(dup_results)}")
print("────────────────────────────────────────────────────────")